# Misc: E-Commerce Scraping (extruct, price-parser, Scrapy)

There is no single "magic" library for e-commerce (like `newspaper` for news), but
there are several tools that help a lot. This notebook covers:

1. **`extruct`** — grab **structured data** (JSON-LD `schema.org/Product`) that e-commerce
   sites often embed. (a **success** & a **failure** example)
2. **`price-parser`** — turn messy price strings of varying formats into numbers.
3. **robots.txt** — what it is, and **good vs bad** scraping practices (compliant vs ignoring it).
4. **`Scrapy`** — a crawling framework: **async engine**, concurrency control (with vs without,
   comparing time), retries, throttling, and `ROBOTSTXT_OBEY`.
5. **`scrapy-poet` + `zyte-common-items`** — the **page object** pattern for clean & reusable code.

Install:

```bash
uv sync --extra ecommerce
```

**Tooling:** `extruct`, `price-parser`, `scrapy` (+ `scrapy-poet`, `zyte-common-items` optional).


## 1. `extruct` — Structured Data (JSON-LD)

Many e-commerce sites embed product data in **schema.org** format (usually JSON-LD)
inside `<script type="application/ld+json">`. The goal is SEO/Google, but it's very
useful for us: **name, price, currency, stock, rating** are already neatly structured —
just grab them, no selectors needed.

`extruct` extracts JSON-LD / Microdata / RDFa / OpenGraph all at once.

### SUCCESS example — a site that provides JSON-LD Product


In [1]:
import extruct

# A GOOD page: embeds a JSON-LD Product (like many real online stores)
html_ok = """
<html><head>
<script type="application/ld+json">
{
  "@context": "https://schema.org",
  "@type": "Product",
  "name": "Gayo Arabica Coffee 250g",
  "brand": "TaniKopi",
  "sku": "COFFEE-250",
  "offers": {
    "@type": "Offer",
    "price": "85000",
    "priceCurrency": "IDR",
    "availability": "https://schema.org/InStock"
  },
  "aggregateRating": {"@type": "AggregateRating", "ratingValue": "4.7", "reviewCount": "128"}
}
</script>
</head><body><h1>Gayo Arabica Coffee</h1></body></html>
"""

data = extruct.extract(html_ok, syntaxes=["json-ld", "microdata", "opengraph"])
product = data["json-ld"][0]  # take the first Product

print("Name     :", product["name"])
print("Brand    :", product["brand"])
print("Price    :", product["offers"]["price"], product["offers"]["priceCurrency"])
print("Stock    :", product["offers"]["availability"])
print("Rating   :", product["aggregateRating"]["ratingValue"])


Name     : Gayo Arabica Coffee 250g
Brand    : TaniKopi
Price    : 85000 IDR
Stock    : https://schema.org/InStock
Rating   : 4.7


### FAILURE example — a site without structured data

Not every site embeds JSON-LD. If there is none, `extruct` returns an **empty** list.
This is where we must **fall back** to the manual approach (BeautifulSoup / XPath). A good pattern:
try `extruct` first (fast & clean), and only parse manually if it comes back empty.


In [2]:
from bs4 import BeautifulSoup

# A BAD page (for us): no structured data at all
html_bad = """
<html><body>
  <div class="product">
    <span class="title">Robusta Coffee 250g</span>
    <span class="price">Rp65.000</span>
  </div>
</body></html>
"""

result = extruct.extract(html_bad, syntaxes=["json-ld", "microdata", "opengraph"])
print("Structured data items found:", {k: len(v) for k, v in result.items()})

# Since it's empty -> fall back to manual parsing
if not result["json-ld"]:
    print("-> No JSON-LD. Falling back to BeautifulSoup.")
    soup = BeautifulSoup(html_bad, "html.parser")
    print("Name :", soup.find("span", class_="title").get_text(strip=True))
    print("Price:", soup.find("span", class_="price").get_text(strip=True))


Structured data items found: {'microdata': 0, 'json-ld': 0, 'opengraph': 0}
-> No JSON-LD. Falling back to BeautifulSoup.
Name : Robusta Coffee 250g
Price: Rp65.000


## 2. `price-parser` — Clean Up Prices

Prices on the web are formatted **chaotically** and differ by country/site: `Rp75.000`,
`$1,234.56`, `€ 1.234,56`, `Rp350.000,-`. Writing your own regex for all these cases is
tiring and error-prone (is a dot a thousands separator or a decimal point?).

`price-parser` handles this automatically: it returns `amount` (a number) + `currency`.


In [3]:
from price_parser import Price

examples = [
    "Rp 1.250.000",
    "Rp75.000",
    "Price: Rp350.000,-",
    "$1,234.56",
    "€ 1.234,56",
    "USD 19.99",
]

print(f"{'input':25} {'amount':>12}  currency")
print("-" * 50)
for s in examples:
    p = Price.fromstring(s)
    print(f"{s:25} {str(p.amount):>12}  {p.currency}")


input                           amount  currency
--------------------------------------------------
Rp 1.250.000                   1250000  Rp
Rp75.000                         75000  Rp
Price: Rp350.000,-              350000  Rp
$1,234.56                      1234.56  $
€ 1.234,56                     1234.56  €
USD 19.99                        19.99  USD


## 3. robots.txt & Scraping Ethics

**`robots.txt`** is a file at the root of a site (e.g. `https://example.com/robots.txt`)
that contains **rules for bots/crawlers**: which parts are allowed (`Allow`) and not allowed
(`Disallow`) to access, and sometimes a `Crawl-delay`. It is a **politeness convention**, not a
technical barrier — but ignoring it can violate the Terms of Service and get your IP blocked.

Python has a built-in reader: `urllib.robotparser`.

In [4]:
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url("https://www.google.com/robots.txt")
try:
    rp.read()
    test_urls = [
        "https://www.google.com/search?q=shoes",  # usually Disallow
        "https://www.google.com/",                 # usually Allow
    ]
    for url in test_urls:
        allowed = rp.can_fetch("*", url)
        print(("ALLOWED" if allowed else "BLOCKED"), "->", url)
except Exception as e:
    print("Failed to read robots.txt:", e)

BLOCKED -> https://www.google.com/search?q=shoes
BLOCKED -> https://www.google.com/


### GOOD vs BAD scraping

| Aspect | ✅ Good | ❌ Bad |
| --- | --- | --- |
| robots.txt | checked & obeyed | ignored |
| User-Agent | clear / honest | empty or disguised |
| Speed | has a delay, ~1 request/second is reasonable | hammer as fast as possible |
| Concurrency | reasonably limited | thousands of parallel requests to 1 site |
| Server load | minimal, take only what's needed | make the site slow/down (DDoS-like) |
| Data | respect ToS & personal data | scrape sensitive data |

Below: a `PoliteFetcher` that **checks robots.txt + adds a delay** before fetching.

In [5]:
import time

import requests
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser


class PoliteFetcher:
    """A 'polite' fetcher: check robots.txt + add a delay + a clear User-Agent."""

    def __init__(self, user_agent="PracticeScrapingBot/1.0 (+learning)", delay=1.0):
        self.ua = user_agent
        self.delay = delay
        self._cache = {}

    def _robot(self, url):
        base = "{0.scheme}://{0.netloc}".format(urlparse(url))
        if base not in self._cache:
            rp = RobotFileParser()
            rp.set_url(base + "/robots.txt")
            try:
                rp.read()
            except Exception:
                rp = None  # no robots.txt -> assume allowed
            self._cache[base] = rp
        return self._cache[base]

    def get(self, url):
        rp = self._robot(url)
        if rp and not rp.can_fetch(self.ua, url):
            raise PermissionError(f"Blocked by robots.txt: {url}")
        time.sleep(self.delay)  # delay so we don't overload the server
        return requests.get(url, headers={"User-Agent": self.ua}, timeout=10)


fetcher = PoliteFetcher(delay=0.5)
resp = fetcher.get("https://books.toscrape.com/")
print("Success (polite):", resp.status_code, "| HTML length:", len(resp.text))

# ❌ The BAD way (DO NOT do this): no robots check, no UA, no delay,
#    and with high concurrency to a single site:
#       for url in thousands_of_urls:
#           requests.get(url)          # spam -> server overwhelmed, IP blocked
print("\n(The bad example is only described, not executed.)")

Success (polite): 200 | HTML length: 51294

(The bad example is only described, not executed.)


## 4. Scrapy — why do we need a framework?

For **real** e-commerce (thousands of products, many pages), `requests` + a plain loop feels
slow and clumsy. `requests.get()` is **synchronous**: one at a time, waiting for the response to
finish before continuing. If each request takes 0.3 seconds, 1000 requests = 5 minutes just waiting.

**Scrapy** is built on an **asynchronous** engine (Twisted/asyncio): many requests
"run concurrently" without waiting on each other (great for I/O work like networking).
On top of that, Scrapy gives you **for free**:

- **Concurrency control**: `CONCURRENT_REQUESTS`, `CONCURRENT_REQUESTS_PER_DOMAIN`.
- **Automatically polite**: `ROBOTSTXT_OBEY`, `DOWNLOAD_DELAY`, `AUTOTHROTTLE` (self-tuning speed).
- **Robust**: automatic retries, timeouts, caching, URL dedup.
- **Pipeline**: clean & store items (e.g. using `price-parser`) in a structured way.

Below we write a single spider for `books.toscrape.com` that grabs the title + price
(cleaned with `price-parser`).

> Note: Scrapy uses the Twisted reactor, which cannot be restarted within a single process,
> so in the notebook we run the spider via a **subprocess** (`scrapy runspider`) — a clean
> approach that matches production.

In [6]:
import tempfile
import textwrap
from pathlib import Path

# Write the spider code to a (.py) file to run it with `scrapy runspider`.
SPIDER_CODE = textwrap.dedent(
    '''
    import scrapy
    from price_parser import Price

    class BooksSpider(scrapy.Spider):
        name = "books"
        # custom_settings = settings specific to this spider
        custom_settings = {
            "ROBOTSTXT_OBEY": True,        # obey robots.txt
            "USER_AGENT": "PracticeScrapingBot/1.0 (+learning)",
            "LOG_LEVEL": "ERROR",          # clean output
            # for polite production, enable AutoThrottle:
            # "AUTOTHROTTLE_ENABLED": True,
        }
        # fetch 12 catalogue pages
        start_urls = [
            f"https://books.toscrape.com/catalogue/page-{i}.html"
            for i in range(1, 13)
        ]

        def parse(self, response):
            for product in response.css("article.product_pod"):
                price_text = product.css("p.price_color::text").get()
                yield {
                    "title": product.css("h3 a::attr(title)").get(),
                    # price-parser: "£51.77" -> 51.77 (float)
                    "price": Price.fromstring(price_text).amount_float,
                }
    '''
)

SPIDER_PATH = str(Path(tempfile.gettempdir()) / "books_spider.py")
Path(SPIDER_PATH).write_text(SPIDER_CODE)
print("Spider written to:", SPIDER_PATH)
print(SPIDER_CODE)

Spider written to: /var/folders/lv/0w17hlfs4073bf5lwh3_79wr0000gn/T/books_spider.py

import scrapy
from price_parser import Price

class BooksSpider(scrapy.Spider):
    name = "books"
    # custom_settings = settings specific to this spider
    custom_settings = {
        "ROBOTSTXT_OBEY": True,        # obey robots.txt
        "USER_AGENT": "PracticeScrapingBot/1.0 (+learning)",
        "LOG_LEVEL": "ERROR",          # clean output
        # for polite production, enable AutoThrottle:
        # "AUTOTHROTTLE_ENABLED": True,
    }
    # fetch 12 catalogue pages
    start_urls = [
        f"https://books.toscrape.com/catalogue/page-{i}.html"
        for i in range(1, 13)
    ]

    def parse(self, response):
        for product in response.css("article.product_pod"):
            price_text = product.css("p.price_color::text").get()
            yield {
                "title": product.css("h3 a::attr(title)").get(),
                # price-parser: "£51.77" -> 51.77 (float)
              

### Async: with vs without concurrency (compare time)

The same spider is run twice, differing only in `CONCURRENT_REQUESTS`:

- `CONCURRENT_REQUESTS=1` → effectively **synchronous** (one request at a time).
- `CONCURRENT_REQUESTS=16` → **async**, many requests running concurrently.

The item count should be the same, but the timing differs greatly. (`DOWNLOAD_DELAY=0` so that
we measure purely the effect of concurrency; in production, still add a delay out of politeness.)

In [7]:
import json
import subprocess
import sys
import time


def run_scrapy(concurrency, out_file):
    t0 = time.perf_counter()
    subprocess.run(
        [
            sys.executable, "-m", "scrapy", "runspider", SPIDER_PATH,
            "-s", f"CONCURRENT_REQUESTS={concurrency}",
            "-s", "DOWNLOAD_DELAY=0",
            "-s", "LOG_LEVEL=ERROR",
            "-O", f"{out_file}:json",  # -O = overwrite the file
        ],
        check=True,
    )
    duration = time.perf_counter() - t0
    count = len(json.load(open(out_file)))
    return duration, count


d1, n1 = run_scrapy(1, "/tmp/scrapy_sync.json")
d16, n16 = run_scrapy(16, "/tmp/scrapy_async.json")

print(f"CONCURRENT_REQUESTS=1  (sync)  : {d1:5.1f} s   ({n1} items)")
print(f"CONCURRENT_REQUESTS=16 (async) : {d16:5.1f} s   ({n16} items)")
print(f"\n=> Async ~{d1 / d16:.1f}x faster for the same amount of data.")

CONCURRENT_REQUESTS=1  (sync)  :   6.2 s   (240 items)
CONCURRENT_REQUESTS=16 (async) :   3.0 s   (240 items)

=> Async ~2.0x faster for the same amount of data.


### Scrapy results → pandas (connects to cleaning & DB)

The Scrapy output (a JSON file) can simply be read into a `DataFrame`. Because the price was already
cleaned with `price-parser` during scraping, the `price` column is numeric right away — ready for the
**cleaning / save-to-database** stage like in the `walkthrough.ipynb` notebook.

In [8]:
import pandas as pd

# read the Scrapy crawl results (from the async run above)
df = pd.read_json("/tmp/scrapy_async.json")

print("DataFrame shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nPrice statistics:")
print(f"  cheapest       : {df['price'].min():.2f}")
print(f"  most expensive : {df['price'].max():.2f}")
print(f"  average        : {df['price'].mean():.2f}")

df.head()

DataFrame shape: (240, 2)

Data types:
title        str
price    float64
dtype: object

Price statistics:
  cheapest       : 10.16
  most expensive : 59.64
  average        : 34.62


,title,price
0,A Light in the Attic,51.77
1,Tipping the Velvet,53.74
2,Soumission,50.10
3,Sharp Objects,47.82
4,Sapiens: A Brief History of Humankind,54.23


## 5. `scrapy-poet` + `zyte-common-items` — clean & reusable code

As a project grows, extraction logic (selectors) mixed inside the spider becomes hard to
maintain. **`scrapy-poet`** introduces the **Page Object** pattern: separate "how to extract
a single page" into its own class. The spider just asks for the result. The benefits:

- **Separated & clean**: selectors live in the Page Object, not scattered across the spider.
- **Reusable & testable**: the Page Object can be reused and tested without running a crawl.
- **Item standard**: **`zyte-common-items`** provides ready-made schemas like `Product`
  (name, price, currency, availability, sku, images, ...) so the output is consistent.
- **Can be combined with Zyte AI extraction** (automatically fills Product fields) for sites
  that have no structured data.

Example pattern (requires a Scrapy project setup + `scrapy-poet` configuration, so it is **not run
in the notebook**):

```python
import attrs
from web_poet import WebPage, field
from zyte_common_items import Product
import scrapy

# Page Object: all extraction logic for a single product page lives here
@attrs.define
class BookProductPage(WebPage):
    @field
    def name(self) -> str:
        return self.css("h1::text").get()

    @field
    def price(self) -> str:
        return self.css("p.price_color::text").get()

    def to_item(self) -> Product:
        return Product(name=self.name, price=self.price, currencyRaw="GBP")

# The spider just ASKS for a Product, knowing nothing about selectors
class BooksSpider(scrapy.Spider):
    name = "books_poet"
    start_urls = ["https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"]

    def parse(self, response, page: BookProductPage):  # page is injected by scrapy-poet
        yield page.to_item()
```

Install (optional): `uv sync --extra ecommerce-advanced`.

## Conclusion — when to use what (e-commerce)

1. **Check structured data first** (`extruct` → JSON-LD `Product`). If present, it's the cleanest.
2. **None there?** Fall back to BeautifulSoup / XPath / a hidden API.
3. **Messy prices?** Clean them up with `price-parser`.
4. **Large scale / many pages?** Use **Scrapy** (async, retry, throttle, robots).
5. **Serious & long-term project?** Tidy it up with **`scrapy-poet`** + **`zyte-common-items`**.
6. **Always** obey `robots.txt`, add delays, use a clear User-Agent, and don't overload the server.

> In short: *start with the lightest & most polite approach, and level up only when you truly need to.*